<a href="https://colab.research.google.com/github/rprieto2809/AIB1/blob/master/pyspark1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ══════════════════════════════════════════════════════════════
# INTRODUCCIÓN A PYSPARK — Google Colab
# Conceptos básicos para Big Data
# ══════════════════════════════════════════════════════════════

# Instalamos PySpark (solo necesario en Colab)
!pip install pyspark -q

# ──────────────────────────────────────────────
# 1. CREAR LA SESIÓN DE SPARK
# ──────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

# SparkSession es el punto de entrada a cualquier programa PySpark
# Es como "arrancar el motor" de Spark
# local[*] significa que usamos todos los núcleos del ordenador disponibles
spark = SparkSession.builder \
    .appName("Introducción a PySpark") \
    .master("local[*]") \
    .getOrCreate()

# Reducimos los mensajes de log para que la salida sea más limpia
spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark arrancado correctamente")
print(f"   Versión de Spark: {spark.version}")
print(f"   Núcleos disponibles: {spark.sparkContext.defaultParallelism}")


# ──────────────────────────────────────────────
# 2. RDD — Resilient Distributed Dataset
# ──────────────────────────────────────────────
print("\n" + "="*55)
print("2. RDD — Resilient Distributed Dataset")
print("="*55)

# Un RDD es la estructura de datos fundamental de Spark
# Es una colección de datos distribuida en múltiples nodos
# 'Resilient' = tolerante a fallos (si un nodo cae, Spark lo recupera)
# 'Distributed' = los datos se reparten entre los núcleos/máquinas

numeros = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# parallelize() divide la lista en particiones que Spark procesa en paralelo
rdd = spark.sparkContext.parallelize(numeros)

print(f"Número de particiones: {rdd.getNumPartitions()}")
# Cada partición se procesa en un núcleo diferente — esto es el PARALELISMO

# collect() recoge todos los datos de vuelta al nodo principal
# ¡Cuidado! En Big Data real nunca usaríamos collect() con millones de registros
print(f"Datos del RDD: {rdd.collect()}")

# count() cuenta los elementos sin traerlos todos al nodo principal
print(f"Número de elementos: {rdd.count()}")


✅ Spark arrancado correctamente
   Versión de Spark: 4.0.2
   Núcleos disponibles: 2

2. RDD — Resilient Distributed Dataset
Número de particiones: 2
Datos del RDD: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Número de elementos: 10


In [ ]:
# ──────────────────────────────────────────────
# 3. MAP — Transformación elemento a elemento
# ──────────────────────────────────────────────
print("\n" + "="*55)
print("3. MAP — Transformación elemento a elemento")
print("="*55)

# map() aplica una función a CADA elemento del RDD
# Es como un bucle for, pero ejecutado en paralelo en todos los núcleos
# La función lambda es una función anónima corta: lambda x: operación

# Elevamos cada número al cuadrado
rdd_cuadrados = rdd.map(lambda x: x ** 2)
print(f"Números originales:  {rdd.collect()}")
print(f"Números al cuadrado: {rdd_cuadrados.collect()}")

# Podemos encadenar varias transformaciones
# filter() filtra los elementos que cumplen una condición
rdd_pares = rdd.filter(lambda x: x % 2 == 0)
print(f"Solo números pares:  {rdd_pares.collect()}")

# flatMap() es como map() pero si la función devuelve una lista,
# la "aplana" — útil para contar palabras en textos
frases = ["hola mundo", "spark es potente", "big data en clase"]
rdd_frases = spark.sparkContext.parallelize(frases)

# Cada frase se divide en palabras y todas se meten en un solo RDD
rdd_palabras = rdd_frases.flatMap(lambda frase: frase.split(" "))
print(f"Palabras extraídas:  {rdd_palabras.collect()}")





In [ ]:
# ──────────────────────────────────────────────
# 4. REDUCE — Agregación distribuida
# ──────────────────────────────────────────────
print("\n" + "="*55)
print("4. REDUCE — Agregación distribuida")
print("="*55)

# reduce() combina todos los elementos del RDD en UN SOLO valor
# La función que le pasamos se aplica de dos en dos hasta reducir todo
# Esto ocurre en paralelo: cada partición reduce sus elementos
# y luego se combinan los resultados de todas las particiones

# Suma de todos los elementos: (((1+2)+3)+4)... en paralelo
suma_total = rdd.reduce(lambda a, b: a + b)
print(f"Suma total:    {suma_total}")

# Máximo: comparamos de dos en dos y nos quedamos con el mayor
maximo = rdd.reduce(lambda a, b: a if a > b else b)
print(f"Valor máximo:  {maximo}")

# Mínimo
minimo = rdd.reduce(lambda a, b: a if a < b else b)
print(f"Valor mínimo:  {minimo}")

# reduceByKey() es muy potente: agrupa por clave y reduce los valores
# Es el corazón del patrón MapReduce (el algoritmo de Google)
ventas = [
    ("Electrónica", 1200), ("Mobiliario", 180),
    ("Electrónica", 350),  ("Mobiliario", 90),
    ("Electrónica", 800),  ("Ropa", 250),
    ("Ropa", 150),         ("Mobiliario", 300),
]
rdd_ventas = spark.sparkContext.parallelize(ventas)

# Sumamos las ventas de cada categoría en paralelo
ventas_por_cat = rdd_ventas.reduceByKey(lambda a, b: a + b)
print(f"\nVentas por categoría: {ventas_por_cat.collect()}")

# sortBy() ordena los resultados
ventas_ordenadas = ventas_por_cat.sortBy(lambda x: x[1], ascending=False)
print(f"Ordenadas por importe: {ventas_ordenadas.collect()}")



In [ ]:
# ──────────────────────────────────────────────
# 5. DATAFRAMES — La forma moderna de usar Spark
# ──────────────────────────────────────────────
print("\n" + "="*55)
print("5. DATAFRAMES — Datos estructurados en Spark")
print("="*55)

# Un DataFrame es como una tabla de base de datos o un Excel
# Es la forma más habitual de trabajar con Spark en la actualidad
# Tiene columnas con nombres y tipos, lo que permite optimizaciones automáticas

# Definimos el esquema (estructura) de los datos
# Esto es opcional pero recomendable para datos grandes
schema = StructType([
    StructField("id",        IntegerType(), False),  # False = no puede ser nulo
    StructField("nombre",    StringType(),  True),
    StructField("categoria", StringType(),  True),
    StructField("ventas",    DoubleType(),  True),
    StructField("unidades",  IntegerType(), True),
    StructField("mes",       IntegerType(), True),
])

# Datos de ejemplo — en Big Data real leeríamos de un CSV, base de datos, etc.
datos = [
    (1,  "Laptop Pro",       "Electrónica", 1200.0, 15, 1),
    (2,  "Silla Ergonómica", "Mobiliario",   180.0, 30, 1),
    (3,  "Monitor 4K",       "Electrónica",  350.0, 20, 2),
    (4,  "Mesa de Oficina",  "Mobiliario",   450.0, 10, 2),
    (5,  "Teclado USB",      "Electrónica",   45.0, 80, 3),
    (6,  "Ratón Inalámbrico","Electrónica",   35.0, 95, 3),
    (7,  "Estantería",       "Mobiliario",   220.0, 12, 4),
    (8,  "Webcam HD",        "Electrónica",   89.0, 40, 4),
    (9,  "Auriculares BT",   "Electrónica",  129.0, 25, 5),
    (10, "Lámpara LED",      "Mobiliario",    55.0, 50, 5),
]

# Creamos el DataFrame
df = spark.createDataFrame(datos, schema=schema)

# show() muestra los datos en forma de tabla (como SELECT * en SQL)
print("DataFrame completo:")
df.show()

# printSchema() muestra la estructura del DataFrame
print("Estructura del DataFrame:")
df.printSchema()


# ──────────────────────────────────────────────
# 6. OPERACIONES CON DATAFRAMES
# ──────────────────────────────────────────────
print("\n" + "="*55)
print("6. OPERACIONES CON DATAFRAMES")
print("="*55)

# SELECT — elegimos las columnas que queremos ver
print("Solo nombre y ventas:")
df.select("nombre", "ventas").show()

# WHERE / FILTER — filtramos filas por condición
print("Productos de Electrónica con ventas > 100€:")
df.filter(
    (F.col("categoria") == "Electrónica") & (F.col("ventas") > 100)
).show()

# withColumn() — añadimos una nueva columna calculada
# Calculamos el ingreso total = ventas * unidades
df = df.withColumn("ingreso_total", F.col("ventas") * F.col("unidades"))
print("Con columna ingreso_total calculada:")
df.select("nombre", "ventas", "unidades", "ingreso_total").show()

# GROUP BY + agregaciones — igual que en SQL
print("Ventas totales e ingresos por categoría:")
df.groupBy("categoria") \
  .agg(
      F.sum("ingreso_total").alias("ingresos_totales"),
      F.avg("ventas").alias("precio_medio"),
      F.count("id").alias("num_productos")
  ) \
  .orderBy("ingresos_totales", ascending=False) \
  .show()

# SQL directo — podemos usar SQL estándar sobre DataFrames
# Primero registramos el DataFrame como tabla temporal
df.createOrReplaceTempView("productos")

print("Consulta SQL directa sobre el DataFrame:")
spark.sql("""
    SELECT categoria,
           SUM(ingreso_total) AS total_ingresos,
           COUNT(*) AS num_productos
    FROM productos
    GROUP BY categoria
    ORDER BY total_ingresos DESC
""").show()




In [ ]:
# ──────────────────────────────────────────────
# 7. ESTADÍSTICAS
# ──────────────────────────────────────────────
print("\n" + "="*55)
print("7. ESTADÍSTICAS DESCRIPTIVAS")
print("="*55)

# describe() calcula automáticamente las estadísticas básicas
# count, mean, stddev, min, max para columnas numéricas
print("Estadísticas descriptivas de ventas e ingresos:")
df.select("ventas", "unidades", "ingreso_total").describe().show()

# Estadísticas específicas
print("Estadísticas manuales:")
stats = df.agg(
    F.mean("ventas").alias("media_ventas"),
    F.stddev("ventas").alias("desv_tipica_ventas"),
    F.percentile_approx("ventas", 0.5).alias("mediana_ventas"),
    F.corr("ventas", "unidades").alias("correlacion_ventas_unidades")
).collect()[0]

print(f"  Media ventas:         {stats['media_ventas']:.2f}€")
print(f"  Desv. típica ventas:  {stats['desv_tipica_ventas']:.2f}€")
print(f"  Mediana ventas:       {stats['mediana_ventas']:.2f}€")
print(f"  Correlación ventas-unidades: {stats['correlacion_ventas_unidades']:.3f}")
# Una correlación negativa significa que los productos más caros
# tienden a venderse en menos unidades — ¡tiene sentido!


# ──────────────────────────────────────────────
# 8. VISUALIZACIONES
# ──────────────────────────────────────────────
print("\n" + "="*55)
print("8. VISUALIZACIONES")
print("="*55)

# Para visualizar con matplotlib necesitamos convertir el DataFrame de Spark
# a un DataFrame de pandas — esto es seguro porque ya tenemos los datos agregados
# En Big Data real: primero agregamos en Spark, luego convertimos a pandas

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        10,
})
COLORS = ['#1B4F72','#17A589','#F39C12','#C0392B','#8E44AD','#2E86C1']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análisis de Ventas con PySpark', fontsize=16, fontweight='bold')

# ── Gráfico 1: Ingresos por categoría (barras)
df_cat = df.groupBy("categoria") \
           .agg(F.sum("ingreso_total").alias("ingresos")) \
           .toPandas()  # convertimos a pandas para graficar

bars = axes[0,0].bar(df_cat['categoria'], df_cat['ingresos'],
                      color=COLORS[:len(df_cat)], edgecolor='white', linewidth=2)
axes[0,0].set_title('Ingresos totales por categoría', fontweight='bold')
axes[0,0].set_ylabel('Ingresos (€)')
for bar, val in zip(bars, df_cat['ingresos']):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                   f'{val:,.0f}€', ha='center', fontweight='bold', fontsize=9)

# ── Gráfico 2: Ventas vs Unidades (scatter — correlación)
df_pd = df.toPandas()  # convertimos todo para el scatter
scatter = axes[0,1].scatter(df_pd['ventas'], df_pd['unidades'],
                             c=COLORS[0], s=120, alpha=0.8, edgecolors='white', linewidth=1.5)
axes[0,1].set_title(f'Correlación Precio vs Unidades\n(r = {stats["correlacion_ventas_unidades"]:.3f})',
                     fontweight='bold')
axes[0,1].set_xlabel('Precio unitario (€)')
axes[0,1].set_ylabel('Unidades vendidas')
for _, row in df_pd.iterrows():
    axes[0,1].annotate(row['nombre'].split()[0],
                        (row['ventas'], row['unidades']),
                        textcoords="offset points", xytext=(5, 5), fontsize=7)

# ── Gráfico 3: Evolución de ingresos por mes
df_mes = df.groupBy("mes") \
           .agg(F.sum("ingreso_total").alias("ingresos")) \
           .orderBy("mes") \
           .toPandas()

axes[1,0].fill_between(df_mes['mes'], df_mes['ingresos'],
                        alpha=0.3, color=COLORS[2])
axes[1,0].plot(df_mes['mes'], df_mes['ingresos'],
               marker='o', color=COLORS[2], linewidth=2.5, markersize=10)
axes[1,0].set_title('Evolución de ingresos por mes', fontweight='bold')
axes[1,0].set_xlabel('Mes')
axes[1,0].set_ylabel('Ingresos (€)')
axes[1,0].set_xticks(df_mes['mes'])
axes[1,0].set_xticklabels([f'Mes {m}' for m in df_mes['mes']])
for x, y in zip(df_mes['mes'], df_mes['ingresos']):
    axes[1,0].text(x, y + 100, f'{y:,.0f}€', ha='center', fontsize=8, fontweight='bold')

# ── Gráfico 4: Top productos por ingreso total (barras horizontales)
df_top = df.orderBy("ingreso_total", ascending=False).toPandas()
bars2 = axes[1,1].barh(df_top['nombre'], df_top['ingreso_total'],
                        color=[COLORS[0] if c == 'Electrónica' else COLORS[1]
                               for c in df_top['categoria']],
                        edgecolor='white', linewidth=1.5)
axes[1,1].set_title('Ranking de productos por ingreso total', fontweight='bold')
axes[1,1].set_xlabel('Ingreso total (€)')
for bar, val in zip(bars2, df_top['ingreso_total']):
    axes[1,1].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                   f'{val:,.0f}€', va='center', fontsize=8, fontweight='bold')
legend_patches = [
    mpatches.Patch(color=COLORS[0], label='Electrónica'),
    mpatches.Patch(color=COLORS[1], label='Mobiliario')
]
axes[1,1].legend(handles=legend_patches, loc='lower right')

plt.tight_layout()
plt.savefig('pyspark_analisis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Notebook completo")
print("\nResumen de conceptos vistos:")
print("  · RDD             → estructura distribuida fundamental de Spark")
print("  · parallelize()   → divide datos en particiones paralelas")
print("  · map()           → transforma cada elemento (en paralelo)")
print("  · filter()        → filtra elementos por condición")
print("  · flatMap()       → map() que aplana listas (útil para texto)")
print("  · reduce()        → agrega todos los elementos en un valor")
print("  · reduceByKey()   → agrega por clave (patrón MapReduce)")
print("  · DataFrame       → tabla estructurada optimizada")
print("  · groupBy/agg     → agrupaciones y estadísticas")
print("  · SQL directo     → consultas SQL sobre DataFrames")
print("  · describe()      → estadísticas descriptivas automáticas")
print("  · toPandas()      → convertir a pandas para visualizar")
# Cerramos la sesión de Spark al terminar
spark.stop()